In [ ]:
import os
import csv
import json
import time
import pathlib
from urllib.parse import urlparse
import requests

# ===== 配置 =====
CSV_PATH = ""                 # CSV路径（表头：package_id,source_code_url）
OUTPUT_DIR = "all_issues_new"             # 输出目录
BATCH_SIZE = 1000                     # 每1000条一个文件
PER_PAGE = 100                        # GitHub最大100
TOKEN = ""  # 建议：export GITHUB_TOKEN=xxxx

if not TOKEN:
    raise SystemExit("请先设置环境变量 GITHUB_TOKEN")

HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"token {TOKEN}",
    "User-Agent": "issue-crawler"
}

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

CURRENT_NDJSON = os.path.join(OUTPUT_DIR, "_current.jsonl")  # 实时保存临时文件
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "_checkpoint.json")  # 保存断点信息


In [2]:
def parse_owner_repo(source_code_url: str):
    """ 从 https://github.com/{owner}/{repo}[.git][/...] 解析 owner, repo """
    try:
        u = urlparse(source_code_url.strip())
        if u.netloc not in {"github.com", "www.github.com"}:
            return None
        parts = [p for p in u.path.strip("/").split("/") if p]
        if len(parts) < 2:
            return None
        owner, repo = parts[0], parts[1]
        if repo.endswith(".git"):
            repo = repo[:-4]
        return owner, repo
    except Exception:
        return None


def respectful_get(url, headers, params, session: requests.Session, max_retries=5):
    """ 带速率限制自我等待 + 退避的 GET """
    backoff = 1
    for attempt in range(max_retries):
        r = session.get(url, headers=headers, params=params, timeout=60)

        remaining = r.headers.get("X-RateLimit-Remaining")
        if r.status_code == 403 and remaining == "0":
            reset_ts = int(r.headers.get("X-RateLimit-Reset", "0"))
            sleep_s = max(1, reset_ts - int(time.time()) + 1)
            print(f"[RateLimit] 剩余=0，等待 {sleep_s}s 至重置...")
            time.sleep(sleep_s)
            continue

        if r.status_code in (429, 403):
            retry_after = r.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                sleep_s = max(1, int(retry_after))
                print(f"[Retry-After] {sleep_s}s")
                time.sleep(sleep_s)
                continue

        if 500 <= r.status_code < 600:
            print(f"[{r.status_code}] 服务端错误，{backoff}s 后重试（第 {attempt+1}/{max_retries} 次）")
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
            continue

        if r.status_code == 403:
            print(f"[403] 可能触发二级限流，{backoff}s 后重试（第 {attempt+1}/{max_retries} 次）")
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
            continue

        if not r.ok:
            r.raise_for_status()

        return r
    raise RuntimeError(f"GET 重试超过 {max_retries} 次仍失败：{url}")


In [3]:
class RollingSaver:
    def __init__(self, out_dir, batch_size):
        self.out_dir = out_dir
        self.batch_size = batch_size
        self.current_path = os.path.join(out_dir, "_current.jsonl")
        self.batch_index = 1
        self.count_in_batch = 0
        self.total = 0
        self._load_checkpoint()

        open(self.current_path, "a", encoding="utf-8").close()

    def _load_checkpoint(self):
        if os.path.exists(CHECKPOINT_PATH):
            with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
                data = json.load(f)
            self.batch_index = data.get("batch_index", 1)
            self.count_in_batch = data.get("count_in_batch", 0)
            self.total = data.get("total", 0)

    def _save_checkpoint(self):
        data = {
            "batch_index": self.batch_index,
            "count_in_batch": self.count_in_batch,
            "total": self.total,
        }
        tmp = self.out_dir + "/._checkpoint.tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f)
        os.replace(tmp, CHECKPOINT_PATH)

    def append_issue(self, obj: dict):
        with open(self.current_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

        self.count_in_batch += 1
        self.total += 1
        self._save_checkpoint()

        if self.count_in_batch >= self.batch_size:
            self._rotate_to_json()

    def _rotate_to_json(self):
        target = os.path.join(self.out_dir, f"issue_{self.batch_index}.json")
        tmp = target + ".tmp"

        items = []
        with open(self.current_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    items.append(json.loads(line))

        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(items, f, indent=2, ensure_ascii=False)

        os.replace(tmp, target)
        print(f"[SAVE] {target}（{len(items)} 条）")

        open(self.current_path, "w", encoding="utf-8").close()
        self.batch_index += 1
        self.count_in_batch = 0
        self._save_checkpoint()

    def finalize(self):
        if self.count_in_batch > 0:
            self._rotate_to_json()
        print(f"[DONE] 共保存 {self.total} 条纯 issue，输出目录：{self.out_dir}")


In [4]:
def crawl_repo_issues(owner, repo, package_id=None, session=None, saver: RollingSaver=None):
    if session is None:
        session = requests.Session()

    page = 1
    base_url = f"https://api.github.com/repos/{owner}/{repo}/issues"

    while True:
        params = {"state": "all", "per_page": PER_PAGE, "page": page}
        r = respectful_get(base_url, HEADERS, params, session)
        data = r.json()

        if not isinstance(data, list):
            print(f"[WARN] {owner}/{repo} 第 {page} 页返回非列表：{data}")
            break
        if not data:
            break

        pure_issues = [it for it in data if "pull_request" not in it]

        for it in pure_issues:
            it["_repo_full_name"] = f"{owner}/{repo}"
            if package_id is not None:
                it["_package_id"] = package_id
            saver.append_issue(it)

        print(f"[{owner}/{repo}] 第 {page} 页：纯 issue {len(pure_issues)}")
        page += 1


In [5]:
saver = RollingSaver(OUTPUT_DIR, BATCH_SIZE)
session = requests.Session()

with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        package_id = row.get("package_id")
        url = row.get("source_code_url")
        parsed = parse_owner_repo(url or "")
        if not parsed:
            print(f"[SKIP] 无法解析 repo：{url}")
            continue
        owner, repo = parsed

        try:
            crawl_repo_issues(owner, repo, package_id, session, saver)
        except Exception as e:
            print(f"[ERROR] 抓取 {owner}/{repo} 出错：{e}")

saver.finalize()


[00-Evan/shattered-pixel-dungeon] 第 1 页：纯 issue 96
[00-Evan/shattered-pixel-dungeon] 第 2 页：纯 issue 88
[00-Evan/shattered-pixel-dungeon] 第 3 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 4 页：纯 issue 100
[00-Evan/shattered-pixel-dungeon] 第 5 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 6 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 7 页：纯 issue 100
[00-Evan/shattered-pixel-dungeon] 第 8 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 9 页：纯 issue 98
[00-Evan/shattered-pixel-dungeon] 第 10 页：纯 issue 99
[SAVE] all_issues_new/issue_1.json（1000 条）
[00-Evan/shattered-pixel-dungeon] 第 11 页：纯 issue 97
[00-Evan/shattered-pixel-dungeon] 第 12 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 13 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 14 页：纯 issue 98
[00-Evan/shattered-pixel-dungeon] 第 15 页：纯 issue 97
[00-Evan/shattered-pixel-dungeon] 第 16 页：纯 issue 98
[00-Evan/shattered-pixel-dungeon] 第 17 页：纯 issue 99
[00-Evan/shattered-pixel-dungeon] 第 18 页：纯 issue 94
[00-Evan/shattered-pixel-dun